In [2]:
# Exercise 6 + 7 : PDF chat bot : Real World RAG Project

from pypdf import PdfReader

reader = PdfReader("os.pdf")
text = ""
for page in reader.pages:
    text += page.extract_text() + "\n"



In [3]:
def chunk_text(text, chunk_size=100, overlap=20):

    words = text.split()
    step = chunk_size - overlap
    chunks = []
    for i in range(0,len(words),step):
        chunk = " ".join(words[i:i+chunk_size])

        if chunk:
            chunks.append(chunk)
    return chunks

chunks = chunk_text(text)

In [4]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

chunk_embeddings = model.encode(chunks)


W0627 20:02:21.867000 7420 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
C:\Users\uniqu\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [14]:
from sklearn.metrics.pairwise import cosine_similarity
        
def retrieve(query, k=3):
    query_embedding = model.encode([query])

    similarities = cosine_similarity(
        query_embedding,
        chunk_embeddings
    )[0]

    top_indices = similarities.argsort()[-k:][::-1]
    retrieved_chunks = []
    for i in top_indices:
        retrieved_chunks.append(chunks[i])
    return retrieved_chunks

In [11]:
def build_prompt(query, retrieved_chunks):
    context = "\n".join(retrieved_chunks)
    prompt = f"""
Context :
{context}

Question :
{query}

Answer using this above context only

"""
    return prompt


In [ ]:
from google import genai

client = genai.Client(api_key="GEMINI_API_KEY")

In [16]:
while True:

    query = input("Ask: ")

    if query == "exit":
        break

    retrieved_chunks = retrieve(query)

    prompt = build_prompt(
        query,
        retrieved_chunks
    )

    # print(retrieved_chunks)

    response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt
    )

    print("Answer : ")
    print(response.text)

Ask:  what is paging?


Answer : 
Based on the context provided, paging is a memory management technique where "the address is just a page number plus offset with no real meaning to the programmer" and it "causes a little internal fragmentation in the last page."


Ask:  exit
